# Script 4: Normalize all technical terms to their canonical form.
The extracted `technical terms` contain many near-duplicates caused by inconsistent terminology in Medium articles and variability in LLM extraction. For example, we might see "machine learning," "ML," "machinelearning," and "Machine Learning" all referring to the same concept.
To create meaningful, consistent tags for analysis, we canonicalize these terms using a hybrid approach that combines fuzzy string matching with LLM-powered term selection.


In [ ]:
import fenic as fc
from dotenv import load_dotenv

load_dotenv()

fc.configure_logging()

config = fc.SessionConfig(
        app_name="medium_curation",
        semantic=fc.SemanticConfig(
            language_models={
                "flash": fc.GoogleGLAModelConfig(
                    model_name="gemini-2.0-flash",
                    rpm=2000,
                    tpm=4_000_000,
                ),
            },
        ),
    )

session = fc.Session.get_or_create(config)

In [3]:
source = session.table("with_features").select("url", "title", "text", "technical_terms")

## Step 1: Load and Clean Technical Terms
- Lowercase extracted terms
- Remove non-alphanumeric characters
- Trim whitespace

In [ ]:
term_col = fc.col("technical_terms")

clean_expr = fc.text.trim(
    fc.text.regexp_replace(
        fc.text.lower(term_col),
        r"[^a-z0-9 ]",
        ""
    )
)

cleaned_terms = (
    source
    .select("url", "technical_terms")
    .filter(fc.array_size(term_col) > 0)
    .explode("technical_terms")
    .with_column("cleaned_term", clean_expr)
    .select("url", "cleaned_term").cache()
)

unique_cleaned_terms = (
    cleaned_terms
    .drop_duplicates(["cleaned_term"])
    .sort("cleaned_term")
    .to_polars()['cleaned_term'].to_list()
)


## Step 2: Group near duplicate extracted terms using simple string similarity 
There are several more "AI-native" ways this can be done.
1. Cross join all entities. For each pair, use Fenic's `semantic.predicate()` for the model to determine whether there's a match or not. Union find.
2. Use simple string heuristics for clear matches (very high/low string similarity match scores) and use `semantic.predicate()` for borderline cases.


In [5]:
from typing import List, Dict
from rapidfuzz.fuzz import token_sort_ratio

def group_terms(terms: List[str], threshold: int = 80) -> List[List[str]]:
    """
    Cluster normalized terms by fuzzy token similarity.

    Args:
        terms (List[str]): List of raw tags (strings).
        threshold (int, optional): Similarity threshold (0-100) for clustering. Defaults to 80.

    Returns:
        List[List[str]]: List of clusters, where each cluster is a list of similar terms.
    """

    clusters: List[List[str]] = []

    for term in set(terms):
        found: bool = False
        for cluster in clusters:
            if any(token_sort_ratio(term, other) >= threshold for other in cluster):
                cluster.append(term)
                found = True
                break
        if not found:
            clusters.append([term])

    return clusters


## Step 3: Use an LLM to determine the canonical name for groups of near duplicate tags

In [6]:
def map_to_canonical_term(clusters: List[List[str]]) -> Dict[str, str]:
    """
    Generate a mapping from individual terms to their canonical representative.

    For each cluster of related terms, this function uses a language model to choose a single canonical term
    that best represents the group. Each term in a cluster is then mapped to that canonical term.

    Args:
        clusters (List[List[str]]): A list of clusters, where each cluster is a list of related term strings.

    Returns:
        Dict[str, str]: A dictionary mapping each term in the input clusters to its canonical representative.
    """
    cluster_df = session.create_dataframe([
        {
            "cluster": ", ".join(cluster),
            "cluster_list": cluster
        }
        for cluster in clusters
    ])

    multi_term_canonical = (
        cluster_df
        # Remove clusters with only one term to avoid unnecessary LLM call.
        .where(fc.array_size(fc.col("cluster_list")) > 1)
        .with_column(
            "canonical_term",
            fc.semantic.map(
                "Choose the best canonical term to represent the following list of related terms: {cluster}. "
                "Respond with only a single term that is clear, concise, and broadly representative."
            )
        )
        .drop("cluster")
        .cache()
    )
    single_term_canonical = cluster_df.where(fc.array_size(fc.col("cluster_list")) == 1).drop("cluster").cache()
    mapping = {}
    for row in multi_term_canonical.to_pylist():
        for term in row["cluster_list"]:
            mapping[term.strip().lower()] = row["canonical_term"].strip().lower()
    for row in single_term_canonical.to_pylist():
        term = row["cluster_list"][0]
        mapping[term.strip().lower()] = term.strip().lower()
    return mapping

## Step 4: Filter out extracted, canonicalized tags that have fewer than 50 article references.

In [ ]:
clusters: List[List[str]] = group_terms(unique_cleaned_terms)
mapping: Dict[str, str] = map_to_canonical_term(clusters)

@fc.udf(return_type=fc.StringType)
def apply_canonical_term_mapping(term: str) -> str:
    return mapping.get(term).lower().strip()

canonical_term_counts = (
    cleaned_terms
    .with_column("canonical_term", apply_canonical_term_mapping(fc.col("cleaned_term")))
    .group_by("canonical_term").agg(fc.count("*").alias("count"))
    .order_by(fc.col("count").desc())
)
popular_terms = set([
    x["canonical_term"]
    for x in (
        canonical_term_counts
        .where(fc.col("count") >= 50)
        .drop("count")
        .to_pylist()
    )
])
filtered_mapping = {k: v for k, v in mapping.items() if v in popular_terms}

## Step 5: Join Tags and Save Results

In [ ]:
@fc.udf(return_type=fc.StringType)
def apply_canonical_term_mapping(term: str) -> str:
    return filtered_mapping.get(term, None)

with_canonical_term = (
    cleaned_terms
    .with_column(
        "canonical_term",
        apply_canonical_term_mapping(fc.col("cleaned_term"))
    )
    .filter(fc.col("canonical_term").is_not_null())
)

with_canonical_term.show()

In [ ]:
with_cleaned_technical_term_list = (
    with_canonical_term
    .group_by("url")
    .agg(
        fc.collect_list("canonical_term").alias("cleaned_terms")
    )
)
with_cleaned_technical_term_list.explode("cleaned_terms").group_by("cleaned_terms").agg(fc.count("*").alias("count")).order_by(fc.col("count").desc()).show(100)

joined = (
    source
    .join(with_cleaned_technical_term_list, on="url", how="left")
    .cache()
)

In [ ]:
joined.write.save_as_table("with_cleaned_technical_terms", mode="overwrite")

In [15]:
session.stop()